In [ ]:
import os
import asyncio
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI     # async client — required by ragas abatch_score



In [ ]:
# ── RAGAS ─────────────────────────────────────────────────────────────────────
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas import SingleTurnSample
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness,
)


load_dotenv()
print("✅ All imports loaded")

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
JUDGE_GROQ_API_KEY = os.getenv("JUDGE_GROQ", GROQ_API_KEY)

# print(GROQ_API_KEY)
groq_client = AsyncOpenAI(
    api_key=JUDGE_GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

judge_llm = llm_factory("llama-3.1-8b-instant", provider="openai", client=groq_client)

# ── Local Embeddings (zero API cost) ──────────────────────────────────────────
ragas_embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    use_api=False,
)

print(f"✅ Judge LLM  : {type(judge_llm).__name__}")
print(f"✅ Embeddings : {type(ragas_embeddings).__name__} (local — no API key needed)")
using = "JUDGE_GROQ" if os.getenv("JUDGE_GROQ") else "GROQ_API_KEY (fallback)"
print(f"✅ Using key  : {using}")


In [ ]:
async def async_cooldown(seconds=60):
    """Async cooldown — works inside Jupyter's running event loop."""
    print(f"\n⏳ Cooldown {seconds}s (Groq rate-limit buffer)...", end=" ")
    for _ in range(seconds // 10):
        await asyncio.sleep(10)
        print(".", end="", flush=True)
    print("  ✅ Ready.\n")
    
    
# scores --  0 - 1

def show_scores(df, col, title):
    """Colour-coded score table for one metric column."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for i, row in df.iterrows():
        score = row.get(col, float("nan"))
        if score != score:
            bar = "⬜"
        elif score >= 0.75:
            bar = "🟢"
        elif score >= 0.5:
            bar = "🟡"
        else:
            bar = "🔴"
        q = str(row.get("user_input", f"Sample {i+1}"))[:55]
        print(f"  {bar}  {score:.2f}  |  {q}")
    avg = df[col].mean()
    label = "✅ Good" if avg >= 0.75 else ("⚠️  Fair" if avg >= 0.5 else "❌ Poor")
    print(f"{'─'*60}")
    print(f"  AVG: {avg:.2f}  {label}")
    print(f"{'='*60}\n")

